In [4]:
import pandas as pd
import string, json, re

## Reorganize the Signalling forms

### Signalling and Interlocking

In [2]:
def flatten_json(data, parent_key="", sep="."):
    items = {}

    if isinstance(data, str):
        try:
            data = json.loads(data.replace("'", '"'))
        except:
            return {}

    if isinstance(data, list):
        for entry in data:
            if isinstance(entry, dict):
                items.update(flatten_json(entry, parent_key, sep))
        return items

    if not isinstance(data, dict):
        return items

    for k, v in data.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k

        if isinstance(v, dict) or isinstance(v, list):
            items.update(flatten_json(v, new_key, sep))
        else:
            items[new_key] = v

    return items

def clean_column_names(df):
    new_columns = []

    for col in df.columns:
        if col.startswith("procedures."):
            col = col.replace("procedures.", "", 1)

        col = re.sub(
            r'(occ_building|dcc_building)\.procedures\.',
            r'\1.',
            col
        )

        new_columns.append(col)

    df.columns = new_columns
    return df


path = "../../output/snc/signalling_system.xlsx" 
df = pd.read_excel(path, sheet_name="occ_dcc", keep_default_na=False)

base_cols = df[["workorder_id", "filename"]]

flattened_list = [flatten_json(row) for row in df['ctc']]

df_flat = pd.DataFrame(flattened_list)

df_out = pd.concat(
    [base_cols, df_flat],
    axis=1
)

df_out = clean_column_names(df_out)

df_out.columns

output_file = f"../../output/snc/signalling_ctc.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

FileNotFoundError: [Errno 2] No such file or directory: '../../output/snc/signalling_system.xlsx'

In [ ]:
def flatten_json(data, parent_key="", sep="."):
    items = {}
    if isinstance(data, str):
        try:
            data = json.loads(data.replace("'", '"'))
        except:
            return {}

    if not isinstance(data, dict):
        return {}

    for k, v in data.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_json(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

path = "../../output/snc/signalling_system.xlsx" 
df = pd.read_excel(path, sheet_name="station", keep_default_na=False)

base_cols = df[["workorder_id", "filename"]]

sig_interlock_flat = pd.DataFrame(
    [flatten_json(row, parent_key="signalling_and_interlocking")
     for row in df["signalling_and_interlocking"]]
)

wayside_flat = pd.DataFrame(
    [flatten_json(row, parent_key="wayside_signalling")
     for row in df["wayside_signalling"]]
)

df_flat = pd.concat(
    [base_cols, sig_interlock_flat, wayside_flat],
    axis=1
)

df_flat.columns

output_file = f"../../output/snc/signalling_station.xlsx"
df_flat.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/snc/response_signalling_station.xlsx


## Flatten into JSON

In [9]:
def to_float(value):
    """
    Converts a value to float. 
    Returns 0.0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return float(clean_val)
    except (ValueError, TypeError):
        return None

def to_int(value):
    """
    Converts a value to int. 
    Returns 0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return int(clean_val)
    except (ValueError, TypeError):
        return None
    
def to_bool(value):
    """
    Converts a value to boolean.
    Returns None if the value is None or empty.
    """
    if pd.isna(value):
        return None
    
    str_val = str(value).strip()
    
    if str_val.upper() == "N/A":
        return "N/A"
    
    if str_val == "":
        return None
    
    str_val = str(value).lower()
    
    if str_val in ['true', '1', 'yes', 'pass']:
        return True
    elif str_val in ['false', '0', 'no', 'notpass']:
        return False
    else:
        return None

def process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns):
    """
    Flattens all columns in the dataframe into a simple key:value JSON structure.
    """
    df = df.rename(columns=rename_columns)
    exclude_from_data = ["workorder_id", "filename"] + exclude_cols

    for _, row in df.iterrows():
        fname = row.get("filename", "unknown")
        wo_id = str(row.get("workorder_id", ""))
        
        if not fname or fname == "nan":
            continue

        if fname not in final_json:
            final_json[fname] = {
                "workorder_id": wo_id,
                "data": {}
            }

        row_flattened_data = {}
        for col in df.columns:
            if col in exclude_from_data:
                continue
            
            value = row[col]
            
            if any(num_col in col for num_col in float_columns):
                row_flattened_data[col] = to_float(value)
            elif any(num_col in col for num_col in int_columns):
                row_flattened_data[col] = to_int(value)
            elif any(bool_col in col for bool_col in bool_columns):
                row_flattened_data[col] = to_bool(value)
            else:
                row_flattened_data[col] = value if pd.notna(value) else None
                
        final_json[fname]["data"].update(row_flattened_data)

    return final_json

### Signaling & Interlocking + Wayside Signaling

In [10]:
path = "../../output/snc/signalling_station.xlsx" 
df = pd.read_excel(path, sheet_name="Sheet1", keep_default_na=False)

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
exclude_cols = []
rename_columns = {}

for col in df.columns:
    if col.endswith(".status"):
        new_col = col.replace(".procedures.", ".", 1)
        rename_columns[col] = new_col
        df = df.rename(columns=rename_columns)        
        bool_columns.append(new_col)
    elif col.endswith(("part_descriptions", "description")):
        exclude_cols.append(col)

exclude_cols.extend([
    "wayside_signalling.station",
    "wayside_signalling.date_time",
    "wayside_signalling.pm_order_no",
    "signalling_and_interlocking.pm_order_no"
])

df = df.rename(columns={
    "signalling_and_interlocking.station" : "station",
    "signalling_and_interlocking.date_time" : "date_time",
    "wayside_signalling.performed_by": "wayside_signalling.technician_id",
    "wayside_signalling.verified_by": "wayside_signalling.supervisor_id",
    "signalling_and_interlocking.performed_by": "signalling_and_interlocking.technician_id",
    "signalling_and_interlocking.verified_by": "signalling_and_interlocking.supervisor_id",
})

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/snc/response_signalling_and_interlocking.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/snc/response_signalling_and_interlocking.xlsx


### CTC

In [11]:
path = "../../output/snc/signalling_ctc.xlsx" 
df = pd.read_excel(path, sheet_name="Sheet1", keep_default_na=False)

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
exclude_cols = []
rename_columns = {}

for col in df.columns:
    if col.endswith(".status"):  
        bool_columns.append(col)
    elif col.endswith(("part_descriptions", "description")):
        exclude_cols.append(col)

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for filename, payload in final_json.items():
    rows.append({
        "filename": filename,
        "workorder_no": payload.get("workorder_id"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/snc/response_signalling_ctc.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/snc/response_signalling_ctc.xlsx
